In [1]:
# !pip install torch
import time, json, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn
from pathlib import Path
warnings.filterwarnings("ignore")
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, matthews_corrcoef, confusion_matrix)

P = Path("processed"); B = P / "balanced"; M = P / "models"; M.mkdir(exist_ok=True)
SEED, EPOCHS, PATIENCE, BS, LR = 42, 40, 5, 512, 1e-3
SETS = ["none", "class_weight", "smote_enn"]

X_val = torch.tensor(np.load(P / "X_val.npy"), dtype=torch.float32)
y_val_np = np.load(P / "y_val.npy")
y_val = torch.tensor(y_val_np, dtype=torch.float32)

In [2]:
def make_mlp(n_features=68):
    return nn.Sequential(
        nn.Linear(n_features, 128), nn.ReLU(),
        nn.Linear(128, 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 1))          # raw score (logit); sigmoid is applied at prediction time

In [3]:
rows, cms, hist = [], {}, {}
for s in SETS:
    torch.manual_seed(SEED); np.random.seed(SEED)
    d = np.load(B / ("none.npz" if s == "class_weight" else f"{s}.npz"))
    X = torch.tensor(d["X"], dtype=torch.float32); y = torch.tensor(d["y"], dtype=torch.float32)

    if s == "class_weight":                      # balanced per-sample weights
        n = len(y)
        w = torch.where(y == 1,
                        torch.tensor(n / (2 * (y == 1).sum().item())),
                        torch.tensor(n / (2 * (y == 0).sum().item())))
    else:
        w = torch.ones(len(y))

    net = make_mlp(X.shape[1])
    opt = torch.optim.Adam(net.parameters(), lr=LR)
    bce = nn.BCEWithLogitsLoss(reduction="none")   # per-sample, so we can apply weights
    val_bce = nn.BCEWithLogitsLoss()               # plain loss for early stopping

    best, bad, best_state, losses, t = 1e9, 0, None, [], time.time()
    for ep in range(EPOCHS):
        net.train(); perm = torch.randperm(len(y))
        for i in range(0, len(y), BS):
            idx = perm[i:i + BS]
            opt.zero_grad()
            loss = (bce(net(X[idx]).squeeze(1), y[idx]) * w[idx]).mean()
            loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            vl = val_bce(net(X_val).squeeze(1), y_val).item()
        losses.append(vl)
        if vl < best - 1e-5:
            best, bad = vl, 0
            best_state = {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
    fit_s = time.time() - t

    net.load_state_dict(best_state); net.eval()
    torch.save(net.state_dict(), M / f"mlp_{s}.pt")

    with torch.no_grad():
        prob = torch.sigmoid(net(X_val).squeeze(1)).numpy()
    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val_np, pred).ravel()
    cms[s] = [[int(tn), int(fp)], [int(fn), int(tp)]]
    rows.append({
        "set": s, "train_rows": len(y),
        "accuracy": accuracy_score(y_val_np, pred),
        "precision": precision_score(y_val_np, pred),
        "recall_attack": recall_score(y_val_np, pred),
        "normal_recall": tn / (tn + fp),
        "f1": f1_score(y_val_np, pred),
        "macro_f1": f1_score(y_val_np, pred, average="macro"),
        "pr_auc": average_precision_score(y_val_np, prob),
        "roc_auc": roc_auc_score(y_val_np, prob),
        "mcc": matthews_corrcoef(y_val_np, pred),
        "FPR": fp / (fp + tn), "FNR": fn / (fn + tp),
        "epochs_run": len(losses), "best_epoch": int(np.argmin(losses)) + 1,
        "best_val_loss": best, "fit_s": fit_s})
    hist[s] = losses
    print(s, "done:", len(losses), "epochs")

results = pd.DataFrame(rows).sort_values("mcc", ascending=False)
results.round(4)

none done: 30 epochs
class_weight done: 11 epochs
smote_enn done: 22 epochs


,set,train_rows,accuracy,precision,recall_attack,normal_recall,f1,macro_f1,pr_auc,roc_auc,mcc,FPR,FNR,epochs_run,best_epoch,best_val_loss,fit_s
0,none,140311,0.9889,0.9893,0.9979,0.9313,0.9936,0.9757,0.9987,0.9930,0.9520,0.0687,0.0021,30,25,0.0439,85.7438
2,smote_enn,236452,0.9886,0.9920,0.9948,0.9490,0.9934,0.9756,0.9990,0.9944,0.9512,0.0510,0.0052,22,17,0.0492,108.2772
1,class_weight,140311,0.9705,0.9830,0.9829,0.8913,0.9829,0.9371,0.9976,0.9864,0.8742,0.1087,0.0171,11,6,0.1037,18.0642
